In [1]:
import pandas as pd
from datetime import datetime, timezone
import os

print("Silver pipeline started...")

Silver pipeline started...


In [2]:
# 1. LOAD DATA


df = pd.read_csv("flights_2023.csv")
print(f"Loaded dataset. Initial rows: {df.shape[0]}")

Loaded dataset. Initial rows: 463484


In [3]:
# 2. STANDARDIZE COLUMN NAMES


df.columns = df.columns.str.strip().str.lower()
print("Column standardization completed.")

Column standardization completed.


In [4]:
# 3. TYPE CASTING


df["fl_date"] = pd.to_datetime(df["fl_date"], errors="coerce")

numeric_columns = [
    "dot_code", "fl_number", "crs_dep_time", "dep_time", "dep_delay",
    "taxi_out", "wheels_off", "wheels_on", "taxi_in",
    "crs_arr_time", "arr_time", "arr_delay",
    "cancelled", "diverted",
    "crs_elapsed_time", "elapsed_time", "air_time", "distance",
    "delay_due_carrier", "delay_due_weather",
    "delay_due_nas", "delay_due_security",
    "delay_due_late_aircraft"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Type casting completed.")

Type casting completed.


In [5]:
# 4. NULL HANDLING


rows_before = df.shape[0]

important_columns = [
    "fl_date",
    "airline",
    "fl_number",
    "origin",
    "dest"
]

df = df.dropna(subset=important_columns)

df["cancelled"] = df["cancelled"].fillna(0)
df["diverted"] = df["diverted"].fillna(0)

rows_after = df.shape[0]
print(f"Null handling completed. Rows removed: {rows_before - rows_after}")


Null handling completed. Rows removed: 0


In [6]:
# 5. REMOVE DUPLICATES


rows_before = df.shape[0]

df = df.drop_duplicates(
    subset=["fl_date", "airline_code", "fl_number", "origin", "dest"]
)

rows_after = df.shape[0]
print(f"Duplicate removal completed. Duplicates removed: {rows_before - rows_after}")


Duplicate removal completed. Duplicates removed: 0


In [7]:
# 6. ADD INGESTION METADATA


df["ingestion_timestamp"] = datetime.now(timezone.utc)
df["data_source"] = "flights_2023_csv"
df["year"] = df["fl_date"].dt.year
df["month"] = df["fl_date"].dt.month
df["day"] = df["fl_date"].dt.day

print("Metadata columns added.")

Metadata columns added.


In [8]:
# 7. VALIDATION CHECKS


if (df["distance"] < 0).any():
    raise ValueError("Validation failed: Negative distance found.")

invalid_cancelled = df[(df["cancelled"] == 1) & (df["air_time"] > 0)]
if not invalid_cancelled.empty:
    print("Warning: Cancelled flights with air_time detected.")

print("Data validation completed.")

Data validation completed.


In [9]:
# 8. SAVE SILVER LAYER


os.makedirs("silver", exist_ok=True)
df.to_parquet("silver/flights_clean.parquet", index=False)


print(f"Silver layer saved successfully. Final rows: {df.shape[0]}")
print("===== SILVER PIPELINE COMPLETED =====")

Silver layer saved successfully. Final rows: 463484
===== SILVER PIPELINE COMPLETED =====
